<a href="https://colab.research.google.com/github/kamranshahid56-tech/rag_based_youtube_summarizer/blob/main/rag_using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

In [2]:
pip install -U youtube-transcript-api langchain-huggingface langchain-text-splitters langchain-community faiss-cpu huggingface_hub

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
# For the LLM (Chat Model)
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
# For the Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

#Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = "HyNa3XXe91c"
ytt_api = YouTubeTranscriptApi()
try:
  transcript_list = ytt_api.fetch(video_id)
  # transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
  transcript = " ".join(snippet.text for snippet in transcript_list)
  print(transcript)
except TranscriptsDisabled:
    print("No captions available for this video.")

#Step 1b - Indexing (Text Splitting)

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

#Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [6]:
from huggingface_hub import login
login()

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="facebook/bart-large-cnn")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['59628088-2d73-4fbb-8fd2-96cdb0fe70e6'])

#Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retriever

In [ ]:
retriever.invoke('Who is Samay Raina?')

#Step 3 - Augmentation

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation",
    huggingfacehub_api_token=""
)

chat_model = ChatHuggingFace(llm=llm)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

#Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

# Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel(
    {
        'context': retriever | RunnableLambda(format_docs),
        'question': RunnablePassthrough()
    }
)

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke("Summarize this video")